# Tema 06: Bokeh

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 45 minutos
**Herramienta principal:** Bokeh
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar el modelo de "gramática de gráficos" de Bokeh: `figure` + `glyphs` + `ColumnDataSource`.
- Construir gráficos interactivos de series de tiempo financieras (línea y velas/candlestick) con herramientas de zoom, pan y hover.
- Vincular ("linkear") dos gráficos para que compartan zoom/desplazamiento (`x_range` compartido).
- Usar `CustomJS` para agregar interactividad en el navegador (sin servidor de Python) con un `Select`.
- Reconocer cuándo Bokeh es preferible frente a Plotly (grandes volúmenes de datos, control fino de herramientas).

## 🧠 Contenido teórico

### 1. ¿Qué es Bokeh y en qué se diferencia de Plotly?

**Bokeh** es otra librería de visualización interactiva para Python (independiente de Plotly), orientada a navegadores web. Conceptualmente es similar a Plotly (ambas generan JavaScript/HTML), pero su filosofía de diseño es distinta:

| | Plotly | Bokeh |
|---|---|---|
| Nivel de abstracción | Alto (`px.line(df, x=, y=)`) | Medio-bajo, estilo "gramática de gráficos" (`p.line(source=...)`) |
| Modelo de datos | DataFrame directo | `ColumnDataSource` (estructura columnar explícita) |
| Fortaleza | Prototipado rápido, mapas, animaciones | Rendimiento con grandes volúmenes de datos, control fino de herramientas, dashboards con `bokeh server` |
| Interactividad sin servidor | Nativa (cliente) | Con `CustomJS` (JavaScript embebido) |
| Interactividad con lógica Python en vivo | Requiere Dash | Nativa con `bokeh serve` (no lo usaremos hoy; Dash ya cubre ese caso de uso) |

**Regla práctica:** si Plotly ya resuelve el caso de uso (la mayoría de las veces), úsalo por su curva de aprendizaje más suave. Bokeh vale la pena cuando necesitas **control muy fino sobre las herramientas de interacción** (por ejemplo paneles financieros tipo TradingView) o **rendimiento** con datasets grandes.

### 2. Los tres bloques de Bokeh

1. **`figure()`**: el lienzo — equivalente al `Figure`/`Axes` de Matplotlib.
2. **Glyphs**: las formas geométricas que dibujas sobre la figura: `p.line(...)`, `p.circle(...)`, `p.vbar(...)`, `p.segment(...)`. Un gráfico de velas, por ejemplo, se construye combinando `segment` (la mecha) y `vbar` (el cuerpo).
3. **`ColumnDataSource` (CDS)**: la estructura de datos central de Bokeh — un diccionario de columnas (muy parecido a un DataFrame) que **alimenta** a los glyphs. Trabajar con un CDS explícito (en vez de pasar el DataFrame directo) es lo que permite después vincular gráficos, usar `CustomJS` y actualizar datos de forma eficiente.

### 3. Herramientas interactivas

Por defecto Bokeh agrega una barra de herramientas con pan, zoom con rueda, zoom de caja y reset. `HoverTool` agrega tooltips personalizados al pasar el mouse; se configuran con una lista de tuplas `(etiqueta, "@columna")`.

### 4. `output_notebook()`

Así como Plotly se muestra automáticamente en la notebook, Bokeh requiere llamar **una vez por sesión** a `output_notebook()` para que las figuras se rendericen dentro de Colab en lugar de intentar abrir un archivo HTML aparte.

## ⚙️ Configuración del entorno

Cargaremos `tema06_acciones_bolsa.csv`: precios diarios sintéticos (apertura, máximo, mínimo, cierre, volumen) de 4 empresas ficticias (2023-2024).

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema06_acciones_bolsa.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema06_acciones_bolsa.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema06_acciones_bolsa.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema06_acciones_bolsa.csv")

df.head()

## 🧭 Práctica guiada

### Paso 0 · Preparar el entorno de Bokeh para Colab

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Select, CustomJS
from bokeh.layouts import column, row
from bokeh.palettes import Category10

output_notebook()  # necesario una vez por sesión para que los gráficos se vean en Colab

df["fecha"] = pd.to_datetime(df["fecha"])

### Paso 1 · Línea simple con `ColumnDataSource`

In [ ]:
tech = df[df["empresa"] == "TECH"].sort_values("fecha")
source_tech = ColumnDataSource(tech)

p1 = figure(x_axis_type="datetime", width=800, height=350,
            title="Precio de cierre — TECH (2023-2024)")
p1.line("fecha", "precio_cierre", source=source_tech, line_width=2, color="#1f77b4")

p1.xaxis.axis_label = "Fecha"
p1.yaxis.axis_label = "Precio de cierre"
show(p1)

### Paso 2 · Agregar `HoverTool`

In [ ]:
hover = HoverTool(
    tooltips=[("Fecha", "@fecha{%F}"), ("Cierre", "@precio_cierre{0.2f}"), ("Volumen", "@volumen{0,0}")],
    formatters={"@fecha": "datetime"},
    mode="vline",
)
p1.add_tools(hover)
show(p1)

### Paso 3 · Gráfico de velas (candlestick)

Un clásico de Bokeh: combina `segment` (mecha) + `vbar` (cuerpo), coloreando en verde las sesiones alcistas y en rojo las bajistas. Filtramos a un periodo corto para que se aprecien las velas.

In [ ]:
periodo = tech[(tech["fecha"] >= "2024-06-01") & (tech["fecha"] <= "2024-08-31")].copy()
alcista = periodo["precio_cierre"] > periodo["precio_apertura"]
bajista = ~alcista
ancho_barra = 18 * 60 * 60 * 1000  # 18 horas en milisegundos (ancho visual de la vela)

p2 = figure(x_axis_type="datetime", width=800, height=400,
            title="Velas (candlestick) — TECH, jun-ago 2024")

p2.segment(periodo["fecha"], periodo["precio_max"], periodo["fecha"], periodo["precio_min"], color="black")
p2.vbar(periodo["fecha"][alcista], ancho_barra, periodo["precio_apertura"][alcista],
        periodo["precio_cierre"][alcista], fill_color="#26a65b", line_color="black")
p2.vbar(periodo["fecha"][bajista], ancho_barra, periodo["precio_apertura"][bajista],
        periodo["precio_cierre"][bajista], fill_color="#e74c3c", line_color="black")

p2.xaxis.axis_label = "Fecha"
p2.yaxis.axis_label = "Precio"
show(p2)

### Paso 4 · Varias líneas con leyenda interactiva (`click_policy`)

In [ ]:
p3 = figure(x_axis_type="datetime", width=850, height=400,
            title="Precio de cierre — todas las empresas")

for empresa, color in zip(sorted(df["empresa"].unique()), Category10[4]):
    datos_empresa = df[df["empresa"] == empresa].sort_values("fecha")
    p3.line(datos_empresa["fecha"], datos_empresa["precio_cierre"],
            legend_label=empresa, color=color, line_width=2)

p3.legend.click_policy = "hide"  # clic en la leyenda oculta/muestra la serie
p3.legend.location = "top_left"
show(p3)

### Paso 5 · Gráficos vinculados (precio + volumen con `x_range` compartido)

In [ ]:
p_precio = figure(x_axis_type="datetime", width=800, height=300, title="TECH — Precio")
p_precio.line("fecha", "precio_cierre", source=source_tech, color="#1f77b4")

p_volumen = figure(x_axis_type="datetime", width=800, height=200,
                    x_range=p_precio.x_range,  # <- comparte el eje X: hacer zoom en uno afecta al otro
                    title="TECH — Volumen")
p_volumen.vbar("fecha", top="volumen", width=20 * 60 * 60 * 1000, source=source_tech, color="#7f7f7f")

show(column(p_precio, p_volumen))

### Paso 6 · Interactividad sin servidor con `CustomJS` (`Select` de empresa)

In [ ]:
datos_por_empresa = {
    empresa: ColumnDataSource(df[df["empresa"] == empresa].sort_values("fecha"))
    for empresa in sorted(df["empresa"].unique())
}

fuente_activa = ColumnDataSource(datos_por_empresa["TECH"].data)

p4 = figure(x_axis_type="datetime", width=800, height=350, title="Selecciona una empresa")
p4.line("fecha", "precio_cierre", source=fuente_activa, line_width=2, color="#d62728")

selector = Select(title="Empresa:", value="TECH", options=sorted(df["empresa"].unique().tolist()))

callback_js = CustomJS(
    args=dict(fuente_activa=fuente_activa, fuentes=datos_por_empresa),
    code="""
    fuente_activa.data = fuentes[cb_obj.value].data;
    fuente_activa.change.emit();
    """,
)
selector.js_on_change("value", callback_js)

show(column(selector, p4))

## ✍️ Práctica independiente

**Ejercicio 1.** Agrega un `HoverTool` al gráfico de volumen del **Paso 5** mostrando fecha y volumen exacto.

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Construye un gráfico de velas (candlestick) igual al del **Paso 3**, pero para la empresa `'BANCO'` y el periodo enero-marzo 2024.

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Añade al gráfico de líneas del **Paso 4** una media móvil de 30 días (`.rolling(30).mean()`) de `precio_cierre` para la empresa `'TECH'`, como una línea punteada adicional (`line_dash='dashed'`).

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Extiende el `Select` con `CustomJS` del **Paso 6** para que, además de la línea de precio, actualice también un título dinámico del gráfico (`p.title.text`) con el nombre de la empresa seleccionada.

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
hover_vol = HoverTool(
    tooltips=[("Fecha", "@fecha{%F}"), ("Volumen", "@volumen{0,0}")],
    formatters={"@fecha": "datetime"},
)
p_volumen.add_tools(hover_vol)
show(column(p_precio, p_volumen))

# Ejercicio 2
banco = df[df["empresa"] == "BANCO"].sort_values("fecha")
periodo_banco = banco[(banco["fecha"] >= "2024-01-01") & (banco["fecha"] <= "2024-03-31")].copy()
alcista_b = periodo_banco["precio_cierre"] > periodo_banco["precio_apertura"]
bajista_b = ~alcista_b
p_banco = figure(x_axis_type="datetime", width=800, height=400, title="Velas — BANCO, ene-mar 2024")
p_banco.segment(periodo_banco["fecha"], periodo_banco["precio_max"], periodo_banco["fecha"], periodo_banco["precio_min"], color="black")
p_banco.vbar(periodo_banco["fecha"][alcista_b], ancho_barra, periodo_banco["precio_apertura"][alcista_b],
             periodo_banco["precio_cierre"][alcista_b], fill_color="#26a65b", line_color="black")
p_banco.vbar(periodo_banco["fecha"][bajista_b], ancho_barra, periodo_banco["precio_apertura"][bajista_b],
             periodo_banco["precio_cierre"][bajista_b], fill_color="#e74c3c", line_color="black")
show(p_banco)

# Ejercicio 3
tech_mm = tech.copy()
tech_mm["media_movil_30"] = tech_mm["precio_cierre"].rolling(30).mean()
p3.line(tech_mm["fecha"], tech_mm["media_movil_30"], color="black", line_dash="dashed",
        legend_label="TECH - media móvil 30d", line_width=2)
show(p3)

# Ejercicio 4
callback_js_reto = CustomJS(
    args=dict(fuente_activa=fuente_activa, fuentes=datos_por_empresa, titulo=p4.title),
    code="""
    fuente_activa.data = fuentes[cb_obj.value].data;
    fuente_activa.change.emit();
    titulo.text = "Precio de cierre - " + cb_obj.value;
    """,
)
selector.js_on_change("value", callback_js_reto)
show(column(selector, p4))

## 🔎 Cierre y puente al siguiente tema

Con Bokeh dominamos series de tiempo interactivas de alto rendimiento. Pero ninguna de las herramientas anteriores está pensada específicamente para **datos geoespaciales** (coordenadas, mapas). En el **Tema 07 (Folium)**, cerraremos el taller construyendo mapas interactivos en Python sobre Leaflet.js.

## 📚 Recursos adicionales

- [Documentación oficial de Bokeh](https://docs.bokeh.org/en/latest/)
- [Guía de primeros pasos (First steps)](https://docs.bokeh.org/en/latest/docs/first_steps.html)
- [Galería de ejemplos oficiales](https://docs.bokeh.org/en/latest/docs/gallery.html)
- [Guía de `ColumnDataSource`](https://docs.bokeh.org/en/latest/docs/user_guide/basic/data.html)
- [Guía de `CustomJS` (interactividad sin servidor)](https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html)